<a href="https://colab.research.google.com/github/Teivak/FaceRecognitionProject/blob/main/2_HW_ArcFace.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ArcFace Loss (Additive Angular Margin Loss)

## Теория ArcFace

В случае с обучением на задачу классификации первая подходящая лосс-функция, которая нам приходит в голову — Cross-Entropy. И на ней действительно можно обучать сеть для распознавания лиц. Но за много лет люди придумали более хитрые трюки, которые делают обучение сети для распознавания лиц более эффективным. Одним из лучших подходов считается ArcFace (Additive Angular Margin).


**Как устроен ArcFace**:

Стандартные SoftMax + кросс-энтропия (CE) выглядят так:

$$L_{CE} = \frac{-1}{N}\sum_1^N \frac{e^{W_{y_i}^{T}x_i + b_{y_i}}}{\sum^n_{j=1}e^{W_j^Tx_i+b_j}},$$

здесь:
- $x_i \in \mathbb{R^d}$ — вектор $i$-го элемента обучающей выборки перед последним полносвязным слоем сети. $y_i$ — класс этого элемента;
- $W_j \in \mathbb{R^d}$ — j-ый столбец матрицы весов последнего слоя сети (т.е. слоя, который производит итоговую классификацю входящего объекта);
- $b_j \in \mathbb{R^d}$ — j-ый элемент вектора байеса последнего слоя сети;
- $N$ — batch size;
- $n$ — количество классов.


Хотя этот лосс работает хорошо, он явным образом не заставляет эмбеддинги $x_i$ элементов, принадлежащих одному классу, быть близкими друг к другу по расстоянию. И не заставляет эмбеддинги элементов, принадлежащих разным классам, быть далеко друг от друга. Все, что хочет этот лосс — чтобы на основе эмбеддингов $x_i$ можно было хорошо классифицировать элементы, никакие ограничений на расстояния между эмбеддингами $x_i$ он не вводит.

Из-за этого у нейросетей для распознавания лиц, которые обучены на обычном CE loss, бывают проблемы с распознаванием лиц, которые сильно отличаются от лиц того же человека разными доп. атрибутами (шляпа/прическа/очки и т.п.). Просто эмбеддинг для таких лиц получается довольно далек по расстоянию от других эмбеддингов лиц этого же человека.

Давайте теперь немного поправим формулу:
- уберем байес последнего слоя, т.е. сделаем $b_j=0$;
- нормализуем веса последнего слоя: ||$W_j$|| = 1;
- нормализуем эмбеддинги: ||$x_i$|| = 1. Перед подачей их на вход последнему слою (т.е. перед умножением на матрицу $W_j$) умножим их на гиперпараметр s. По сути, мы приводим норму всех эмбеддингов к s. Смысл этого гиперпараметра в том, что, возможно, сети проще будет классифицировать эмбеддинги, у которых не единичная норма.

Нормализация приводит к тому, что эмбеддинги распределяются по сфере единичного радиуса (и сфере радиуса s после умножения на гиперпараметр s). И итоговые предсказания сети после последнего слоя зависят только от угла между эмбеддингами $x_i$ и выученных весов $W_j$. От нормы эмбеддинга $x_i$ они больше не зависят, т.к. у всех эмбеддингов они теперь одинаковые.

Получается, в степени экспоненты у нас останется выражение $s W_{y_i}^{T}x_i$, которое можно переписать в виде  $s W_{y_i}^{T}x_i = s ||W_{y_i}||\cdot ||x_i|| \cdot cos\Theta_{y_i}$. Тут $\Theta_{y_i}$ — это угол между векторами $W_{y_i}$ и $x_i$. Но так как мы сделали нормы $W_{y_i}$ и $x_i$ единичными, то все это выражение просто будет равно $s cos\Theta_{y_i}$.

В итоге мы получим следующую формулу лосса:

$$L = \frac{-1}{N}\sum_1^N \frac{e^{s\ cos\Theta_{y_i}}}{e^{s\ cos\Theta_{y_i}} + \sum^n_{j=1,\ j\ne y_i} e^{s\ cos\Theta_j}}$$


И последний шаг. Добавим еще один гиперпараметр $m$. Он называется additive angular margin penalty и заставляет эмбеддинги одного класса быть ближе друг к другу, а эмбеддинги разных классов — более далекими друг от друга.

В итоге получим вот что:

$$L_{ArcFace} = \frac{-1}{N}\sum_1^N \frac{e^{s\ cos(\Theta_{y_i} + m)}}{e^{s\ cos(\Theta_{y_i} + m)} + \sum^n_{j=1,\ j\ne y_i} e^{s\ cos\Theta_j}}$$

Это и есть ArcFace Loss с двумя  гиперпараметрами, s и m.

Получается, что ArcFace Loss завтавляет сеть выучивать эмбеддинги, распределенные по сфере радиуса s, причем чтобы эмбеддинги одного класса были ближе друг к другу, а эмбеддинги разных классов — более далеки друг от друга.

![ArcFace](https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTKRR-YA_XR3yhIYBbkc8Zlbua0Q2WdM3gx_g&s)

**Важное пояснение:**

Строго говоря, ArcFace - не лосс, отдельный архитектурный модуль модификация SoftMax. Он реализует идею внесения геометрического отступа непосредственно в пространство признаков. Для обучения в качестве лосса используется обычная кросс-энтропия. Более конкретно по шагам:

1. Вы извлекаете эмбеддинги из бэкбона сети (предобученной модели, у которой обрезан FC-слой, если он был)
2. Эти эмбеддинги поступают в ArcFace-слой, который содержит векторы-центры для каждого класса (веса классификатора) и логику нормализации и добавления углового отступа
3. Для целевого класса ArcFace-слой преобразует косинус угла $\theta$ в $cos(\theta + m)$
4. Для остальных классов оставляет обычный косинус $cos(\theta)$
5. Эти модифицированные логиты подаются на вход стандартной функции Cross-Entropy
6. Градиенты от Cross-Entropy текут назад через ArcFace-слой к бэкбону, обучая модель извлекать эмбеддинги

Результат: модифицированные логиты с "жестким" разделением для целевого класса, а значит и более качественные эмбеддинги.

Схема:
```
[Изображение] → [Бэкбон] → [ЭМБЕДДИНГ] → [ArcFace] → [Логиты] → [CE Loss]
                    │                        │           │          
                   CNN                   Нормализация   Оценки
                                          + Angular    для всех
                                            Margin     классов
```

Для получения качественных эмбеддингов после обучения ArcFace-слой больше не нужен, и его обычно обрезают. Он нужен был только обучения модели, и поэтому часто ArcFace называю именно лоссом. Но стоит всегда держать в голове, что это некоторое упрощение, которое нужно лишь для того, чтобы проще формулировать мысли.

**Доп. литература по ArcFace Loss:**

Оригинальная статья: https://arxiv.org/pdf/1801.07698.pdf

## Другие лоссы

Кроме ArcFace, есть еще много разных вариантов лоссов для задачи Face Recognition. Некоторые из них можно найти, например, [тут](https://openaccess.thecvf.com/content_CVPRW_2020/papers/w48/Hsu_A_Comprehensive_Study_on_Loss_Functions_for_Cross-Factor_Face_Recognition_CVPRW_2020_paper.pdf). Вы можете попробовать реализовать другие лосс-функции в этом проекте в качестве дополнительного задания.

Кроме этого, можно миксовать лосс-функции. Например, обучать нейросеть на сумме ArcFace и TripletLoss. Иногда так выходит лучше, чем если обучать на каком-то одном лоссе.

# Датасет

В качестве датасета нужно использовать картинки из CelebA, выровненные при помощи своей модели из задания 1. Очень желательно их еще кропнуть таким образом, чтобы нейросети поступали на вход преимущественно только лица без какого либо фона, частей тела и прочего.

Если планируете делать дополнительное задание на Identificaton rate metric, то **обязательно разбейте заранее датасет на train/val или train/val/test.** Это нужно сделать не только на уровне кода, а на уровне папок, чтобы точно знать, на каких картинках модель обучалась, а на каких нет. Лучше заранее почитайте [ноутбук с заданием](https://colab.research.google.com/drive/15zuNdOupRFnG7oE-rFj9FsjoNTK6DYn5).

# План заданий

Итак, вот, что от вас требуется в этом задании:

* Выбрать модель (или несколько моделей) для обучения. Можно брать предобученные на ImageNet, но нельзя использовать модели, предобученные на задачу распознавания лиц.
* Обучить эту модель (модели) на CE loss. Добиться accuracy > 0.7.
* Реализовать ArcFace loss.
* Обучить модель (модели) на ArcFace loss. Добиться accuracy > 0.7.
* Написать небольшой отчет по обучению, сравнить CE loss и ArcFace loss.

**P.S. Не забывайте сохранять модели после обучения**

In [2]:
import pandas as pd

train_dataset_df = pd.read_csv('PROJECT/FaceAlignment/train_dataset.csv')
val_dataset_df = pd.read_csv('PROJECT/FaceAlignment/val_dataset.csv')
test_dataset_df = pd.read_csv('PROJECT/FaceAlignment/test_dataset.csv')

In [4]:
%run PROJECT/FaceAlignment/FaceRecognitionDataset.py

In [6]:
import kagglehub
import os
import pandas as pd

# Download latest version
path = kagglehub.dataset_download("kevinpatel04/celeba-original-wild-images")

landmarks = pd.read_csv('PROJECT/FaceAlignment/pred_landmarks.csv')
bboxes = pd.read_csv(f'{path}/list_bbox_celeba.csv')

# Helper function to merge a dataset dataframe with landmarks and bboxes
def merge_datasets(dataset_df, landmarks_df, bboxes_df):
    merged_df = pd.merge(dataset_df, landmarks_df, on='image_id', how='left')
    merged_df = pd.merge(merged_df, bboxes_df, on='image_id', how='left')
    return merged_df

# Merge train, val, and test dataframes with landmarks and bboxes
full_train_dataset_df = merge_datasets(train_dataset_df, landmarks, bboxes)
full_val_dataset_df = merge_datasets(val_dataset_df, landmarks, bboxes)
full_test_dataset_df = merge_datasets(test_dataset_df, landmarks, bboxes)

In [ ]:
fixed_image_size = (128, 128) # Определяем фиксированный размер для всех изображений

train_dataset = FaceRecognitionDataset(full_train_dataset_df, path, target_image_size=fixed_image_size)
val_dataset = FaceRecognitionDataset(full_val_dataset_df, path, target_image_size=fixed_image_size)
test_dataset = FaceRecognitionDataset(full_test_dataset_df, path, target_image_size=fixed_image_size)

batch_size = 16
dataloader = DataLoader(face_landmarks_dataset, batch_size=batch_size, shuffle=True, collate_fn=custom_collate_fn, num_workers=0) # Создаём даталоудер

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
import math

class ArcFace(nn.Module):
    """ Implement ArcFace (Additive Angular Margin Loss) """
    def __init__(self, in_features, out_features, s=64.0, m=0.50):
        super(ArcFace, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s  # Scale factor
        self.m = m  # Angular margin

        # Weights (class centers) for each class
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

        # Precompute constants for performance
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        # Threshold for `cos(theta)` when `theta + m` goes beyond `pi`
        self.th = math.cos(math.pi - m)
        # Penalty term for `theta > pi - m` cases
        self.mm = math.sin(math.pi - m) * m

    def forward(self, input_features, labels):
        # Normalize input features and weights to unit vectors
        # input_features: (batch_size, in_features)
        # weights: (out_features, in_features)
        # Normalized features (embeddings) and weights will lie on a unit hypersphere.
        norm_input_features = F.normalize(input_features)
        norm_weights = F.normalize(self.weight)

        # Calculate cosine similarity between normalized features and normalized weights
        # This gives `cos(theta_j)` for each class `j`
        # cosine: (batch_size, out_features)
        cosine = F.linear(norm_input_features, norm_weights)

        # Calculate `sin(theta)` needed for `cos(theta + m)`
        # Clamp values to avoid NaN from sqrt due to floating point inaccuracies
        sine = torch.sqrt(torch.clamp(1.0 - torch.pow(cosine, 2), 0, 1.0))

        # Calculate `cos(theta + m) = cos(theta)cos(m) - sin(theta)sin(m)`
        phi = cosine * self.cos_m - sine * self.sin_m

        # Apply the margin penalty to the target class logits
        # If `cos(theta)` is less than `self.th`, it means `theta > pi - m`.
        # In this case, `theta + m > pi`, and `cos(theta + m)` would behave non-monotonically.
        # To fix this, a linear penalty `cosine - self.mm` is applied instead.
        final_phi = torch.where(cosine > self.th, phi, cosine - self.mm)

        # Create a one-hot mask to identify the target class for each sample
        # one_hot: (batch_size, out_features)
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1)

        # Replace the `cos(theta_yi)` with `cos(theta_yi + m)` for the target class
        # (and adjust for the `theta > pi - m` case with `final_phi`)
        # For other classes, keep `cos(theta_j)`
        output = (one_hot * final_phi) + ((1.0 - one_hot) * cosine)

        # Scale the final logits by `s`
        output *= self.s

        return output


class FaceRecognition(nn.Module):
    """ Combined model for face recognition using a backbone and ArcFace layer. """
    def __init__(self, num_classes, embedding_size=512, s=64.0, m=0.50):
        super(FaceRecognitionModel, self).__init__()

        # 1. Backbone: Use a pre-trained ResNet-50 as a feature extractor.
        # The text specifies using models pre-trained on ImageNet.
        self.backbone = models.resnet50(pretrained=True)

        # Remove the original fully connected (classification) layer of ResNet-50.
        # The layer before the FC layer (avgpool) outputs 2048 features for ResNet-50.
        self.backbone.fc = nn.Identity() # Replaces the FC layer with an identity module

        # 2. Embedding Layer: Project features from the backbone to the desired embedding_size.
        # This layer produces the embeddings (x_i in ArcFace formula).
        self.embedding_layer = nn.Linear(2048, embedding_size)

        # 3. ArcFace Layer: Applies the angular margin penalty for training.
        # It takes the embeddings and labels to produce modified logits for Cross-Entropy loss.
        self.arcface_layer = ArcFace(embedding_size, num_classes, s, m)

    def forward(self, x, labels=None):
        # Pass input image through the backbone to extract raw features.
        features = self.backbone(x)

        # Project raw features to the specified embedding_size.
        embeddings = self.embedding_layer(features)

        # During training, `labels` will be provided. The ArcFace layer will modify logits.
        # During inference (when `labels` is None), we return the normalized embeddings directly.
        # As per the text: "Для получения качественных эмбеддингов после обучения ArcFace-слой больше не нужен, и его обычно обрезают."
        if labels is not None:
            logits = self.arcface_layer(embeddings, labels)
            return logits
        else:
            # For inference, return the L2-normalized embeddings.
            return F.normalize(embeddings)

# Task
```python
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
import math

class ArcFace(nn.Module):
    """ Implement ArcFace (Additive Angular Margin Loss) """
    def __init__(self, in_features, out_features, s=64.0, m=0.50):
        super(ArcFace, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s  # Scale factor
        self.m = m  # Angular margin

        # Weights (class centers) for each class
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

        # Precompute constants for performance
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        # Threshold for `cos(theta)` when `theta + m` goes beyond `pi`
        self.th = math.cos(math.pi - m)
        # Penalty term for `theta > pi - m` cases
        self.mm = math.sin(math.pi - m) * m

    def forward(self, input_features, labels):
        # Normalize input features and weights to unit vectors
        # input_features: (batch_size, in_features)
        # weights: (out_features, in_features)
        # Normalized features (embeddings) and weights will lie on a unit hypersphere.
        norm_input_features = F.normalize(input_features)
        norm_weights = F.normalize(self.weight)

        # Calculate cosine similarity between normalized features and normalized weights
        # This gives `cos(theta_j)` for each class `j`
        # cosine: (batch_size, out_features)
        cosine = F.linear(norm_input_features, norm_weights)

        # Calculate `sin(theta)` needed for `cos(theta + m)`
        # Clamp values to avoid NaN from sqrt due to floating point inaccuracies
        sine = torch.sqrt(torch.clamp(1.0 - torch.pow(cosine, 2), 0, 1.0))

        # Calculate `cos(theta + m) = cos(theta)cos(m) - sin(theta)sin(m)`
        phi = cosine * self.cos_m - sine * self.sin_m

        # Apply the margin penalty to the target class logits
        # If `cos(theta)` is less than `self.th`, it means `theta > pi - m`.
        # In this case, `theta + m > pi`, and `cos(theta + m)` would behave non-monotonically.
        # To fix this, a linear penalty `cosine - self.mm` is applied instead.
        final_phi = torch.where(cosine > self.th, phi, cosine - self.mm)

        # Create a one-hot mask to identify the target class for each sample
        # one_hot: (batch_size, out_features)
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1)

        # Replace the `cos(theta_yi)` with `cos(theta_yi + m)` for the target class
        # (and adjust for the `theta > pi - m` case with `final_phi`)
        # For other classes, keep `cos(theta_j)`
        output = (one_hot * final_phi) + ((1.0 - one_hot) * cosine)

        # Scale the final logits by `s`
        output *= self.s

        return output


class CEHead(nn.Module):
    """ Standard Cross-Entropy Head (a simple linear layer). """
    def __init__(self, embedding_size, num_classes):
        super(CEHead, self).__init__()
        self.fc = nn.Linear(embedding_size, num_classes)

    def forward(self, input_features, labels=None):
        # Labels are not used for CEHead directly, but kept for API consistency.
        return self.fc(input_features)


class AdaCos(nn.Module):
    """ AdaCos (Adaptive Cosine) Head Placeholder.
        Currently implements a simple linear layer, as a placeholder for full AdaCos implementation.
    """
    def __init__(self, embedding_size, num_classes):
        super(AdaCos, self).__init__()
        # In a full implementation, this would involve more complex logic
        # including adaptive scaling and dynamic margin.
        print("Note: AdaCos Head is a placeholder and currently implements a simple linear layer.")
        self.fc = nn.Linear(embedding_size, num_classes)

    def forward(self, input_features, labels=None):
        # Labels are not used for AdaCos directly, but kept for API consistency.
        return self.fc(input_features)


class FaceRecognition(nn.Module):
    """
    Combined model for face recognition using a customizable backbone and classification head.

    Args:
        num_classes (int): Number of identity classes.
        embedding_size (int): Dimension of the embedding features.
        s (float): Scale factor for ArcFace.
        m (float): Angular margin for ArcFace.
        backbone_type (str): Type of backbone to use ('resnet50', 'efficientnet_b0').
        head_type (str): Type of classification head to use ('arcface', 'cehead', 'adacos').
    """
    def __init__(self, num_classes, embedding_size=512, s=64.0, m=0.50,
                 backbone_type='resnet50', head_type='arcface'):
        super(FaceRecognition, self).__init__()

        self.embedding_size = embedding_size
        self.num_classes = num_classes
        self.backbone_type = backbone_type
        self.head_type = head_type
        self.s = s
        self.m = m

        # 1. Load Backbone
        self.backbone, backbone_out_features = self._load_backbone(backbone_type)

        # 2. Embedding Layer: Project features from the backbone to the desired embedding_size.
        self.embedding_layer = nn.Linear(backbone_out_features, embedding_size)

        # 3. Load Classification Head
        self.classification_head = self._load_head(head_type, embedding_size, num_classes, s, m)

    def _load_backbone(self, backbone_type):
        """ Dynamically loads the specified backbone model and its output feature size. """
        if backbone_type == 'resnet50':
            backbone = models.resnet50(pretrained=True)
            # Remove the original fully connected (classification) layer.
            # The layer before the FC layer (avgpool) outputs 2048 features for ResNet-50.
            backbone.fc = nn.Identity()
            out_features = 2048
        elif backbone_type == 'efficientnet_b0':
            backbone = models.efficientnet_b0(pretrained=True)
            # EfficientNet's classifier is a sequential block; we need features before it.
            # The `in_features` of the linear layer (index 1) in the classifier is the output size.
            out_features = backbone.classifier[1].in_features
            backbone.classifier = nn.Identity() # Remove the entire classifier block
        else:
            raise ValueError(f"Unsupported backbone type: {backbone_type}. Choose from 'resnet50', 'efficientnet_b0'.")
        return backbone, out_features

    def _load_head(self, head_type, embedding_size, num_classes, s, m):
        """ Dynamically loads the specified classification head. """
        if head_type == 'arcface':
            return ArcFace(embedding_size, num_classes, s, m)
        elif head_type == 'cehead':
            return CEHead(embedding_size, num_classes)
        elif head_type == 'adacos':
            return AdaCos(embedding_size, num_classes)
        else:
            raise ValueError(f"Unsupported head type: {head_type}. Choose from 'arcface', 'cehead', 'adacos'.")

    def forward(self, x, labels=None):
        # Pass input image through the backbone to extract raw features.
        features = self.backbone(x)

        # Project raw features to the specified embedding_size.
        embeddings = self.embedding_layer(features)

        # During training, `labels` will be provided. The classification head will modify logits.
        # During inference (when `labels` is None), we return the normalized embeddings directly.
        # As per the text: "Для получения качественных эмбеддингов после обучения ArcFace-слой больше не нужен, и его обычно обрезают."
        if labels is not None:
            logits = self.classification_head(embeddings, labels)
            return logits
        else:
            # For inference, return the L2-normalized embeddings.
            return F.normalize(embeddings)

```

## Modify FaceRecognition for multiple backbones and heads

### Subtask:
Refactor the `FaceRecognition` class to dynamically load specified backbones (ResNet, EfficientNet) and integrate different classification heads (ArcFace, CEHead, AdaCos placeholder). Implement a `CEHead` class for standard Cross-Entropy and a basic `AdaCos` class as a placeholder.


## Summary:

### Data Analysis Key Findings
*   A flexible `FaceRecognition` model has been developed, allowing dynamic selection of backbones and classification heads.
*   The model supports two pre-trained backbones: `resnet50` and `efficientnet_b0`, with their original classification layers replaced by an `nn.Identity()` to extract features.
*   Three types of classification heads are integrated:
    *   **ArcFace:** A complete implementation of the Additive Angular Margin Loss, including normalization of features and weights, margin `m`, scale `s`, and handling of the `theta + m > pi` condition for robust training.
    *   **CEHead:** A standard Cross-Entropy head, implemented as a simple linear layer.
    *   **AdaCos:** Included as a placeholder, currently functioning as a basic linear layer, indicating it awaits a full implementation of its adaptive scaling and dynamic margin features.
*   An intermediate `embedding_layer` (a linear transformation) is used to project the backbone's output features to a specified `embedding_size` before passing them to the classification head.
*   The `forward` method is designed to differentiate between training and inference: during training (when labels are provided), it returns logits from the chosen classification head; during inference (when labels are `None`), it returns L2-normalized embeddings, which are typically used for similarity comparisons.

### Insights or Next Steps
*   **Complete AdaCos Implementation:** Prioritize completing the full implementation of the AdaCos head to leverage its adaptive scaling and dynamic margin properties, potentially improving performance over fixed-margin methods like ArcFace.
*   **Performance Evaluation and Hyperparameter Tuning:** Systematically evaluate the performance of different backbone and head combinations (e.g., ResNet50 with ArcFace, EfficientNet with ArcFace/CEHead) on a relevant dataset to identify the most effective architecture and tune hyperparameters like `s` and `m` for optimal results.
